# Topic Modeling

## What is Topic Modeling?
Unsupervised method to discover hidden thematic structure in a collection of documents. No labels required.

**Applications**: News categorization, document organization, trend discovery, recommendation.

---

## 1. LDA (Latent Dirichlet Allocation)

### Generative Process
For each document $d$:
1. Choose topic distribution: $\theta_d \sim \text{Dir}(\alpha)$
2. For each word position $n$:
   - Choose topic: $z_{d,n} \sim \text{Multinomial}(\theta_d)$
   - Choose word: $w_{d,n} \sim \text{Multinomial}(\beta_{z_{d,n}})$
   - where $\beta_k \sim \text{Dir}(\eta)$ is the word distribution for topic $k$

### Dirichlet Distribution
$$\text{Dir}(\alpha) = \frac{\Gamma\left(\sum_k \alpha_k\right)}{\prod_k \Gamma(\alpha_k)} \prod_k \theta_k^{\alpha_k - 1}$$

- Low $\alpha$: Documents are focused on few topics
- High $\alpha$: Documents spread over many topics

### Joint Distribution
$$P(w, z, \theta, \beta | \alpha, \eta) = \prod_k P(\beta_k | \eta) \prod_d P(\theta_d | \alpha) \prod_n P(z_{d,n} | \theta_d) P(w_{d,n} | \beta_{z_{d,n}})$$

### Inference
Exact inference is intractable. Use:
- **Variational Inference**: Approximate $P(z|w)$ with a simpler distribution
- **Collapsed Gibbs Sampling**: Sample $z_{d,n}$ given all other assignments:

$$P(z_{d,n} = k | \mathbf{z}_{-dn}, \mathbf{w}) \propto \frac{n_{d,k}^{-dn} + \alpha_k}{\sum_k n_{d,k}^{-dn} + \alpha_k} \cdot \frac{n_{k,w}^{-dn} + \eta_w}{\sum_w n_{k,w}^{-dn} + \sum_w \eta_w}$$

---

## 2. NMF (Non-negative Matrix Factorization)

Factorize document-term matrix $V \approx WH$ where $W, H \geq 0$:

$$\min_{W, H \geq 0} ||V - WH||_F^2$$

$W$ = document-topic matrix, $H$ = topic-term matrix.

---

## 3. LSA/LSI (Latent Semantic Analysis)

Apply SVD to the TF-IDF matrix: $A = U \Sigma V^T$

Truncate to top-$k$ components to discover latent semantic structure.

---

## 4. BERTopic

Modern neural topic model:
1. **Embed** documents with sentence transformers
2. **Reduce** dimensionality with UMAP
3. **Cluster** with HDBSCAN
4. **Represent** topics with c-TF-IDF:

$$c\text{-TF-IDF}_{t,c} = \text{TF}_{t,c} \times \log\left(1 + \frac{A}{\text{tf}_t}\right)$$

Where $c$ is a cluster (not a document), $A$ = average number of words per class.

---

## 5. Evaluation

- **Coherence Score (CV)**: Measures semantic similarity of top words per topic (higher is better)
- **Perplexity**: Lower is better (but can diverge from human judgment)
- **Topic Diversity**: Unique words across topics / total words

In [1]:
import numpy as np
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import NMF, LatentDirichletAllocation, TruncatedSVD

# Load dataset
categories = ['comp.graphics', 'sci.space', 'rec.sport.hockey', 'talk.politics.guns']
newsgroups = fetch_20newsgroups(subset='train', categories=categories,
                                 remove=('headers', 'footers', 'quotes'))
texts = newsgroups.data[:500]  # Use subset for speed
print(f"Loaded {len(texts)} documents from {len(categories)} categories")

Loaded 500 documents from 4 categories


In [2]:
# ============================================================
# LDA
# ============================================================
def print_topics(model, feature_names, n_top_words=10):
    for i, topic in enumerate(model.components_):
        top_words = [feature_names[j] for j in topic.argsort()[-n_top_words:][::-1]]
        print(f"  Topic {i+1}: {', '.join(top_words)}")

# Vectorize
count_vec = CountVectorizer(max_df=0.9, min_df=2, stop_words='english', max_features=1000)
X_count = count_vec.fit_transform(texts)
feature_names = count_vec.get_feature_names_out()

n_topics = 4

# LDA
lda = LatentDirichletAllocation(
    n_components=n_topics,
    max_iter=20,
    learning_method='online',
    random_state=42
)
lda.fit(X_count)

print("LDA Topics:")
print_topics(lda, feature_names)
print(f"\nLDA Perplexity: {lda.perplexity(X_count):.2f}")

LDA Topics:
  Topic 1: like, just, think, time, game, don, team, people, good, year
  Topic 2: space, nasa, earth, orbit, moon, mars, shuttle, lunar, probe, launch
  Topic 3: file, gun, firearms, control, law, congress, states, house, mr, weapons
  Topic 4: gun, guns, use, don, image, people, edu, like, program, just



LDA Perplexity: 611.27


In [3]:
# ============================================================
# NMF
# ============================================================
tfidf_vec = TfidfVectorizer(max_df=0.9, min_df=2, stop_words='english', max_features=1000)
X_tfidf = tfidf_vec.fit_transform(texts)
feature_names_tfidf = tfidf_vec.get_feature_names_out()

nmf = NMF(n_components=n_topics, random_state=42, max_iter=500)
nmf.fit(X_tfidf)

print("NMF Topics:")
print_topics(nmf, feature_names_tfidf)

NMF Topics:
  Topic 1: gun, guns, people, don, just, think, right, government, like, weapons
  Topic 2: thanks, files, file, know, 3d, image, does, software, help, advance
  Topic 3: space, nasa, shuttle, launch, moon, orbit, earth, program, lunar, mars
  Topic 4: game, hockey, team, season, players, year, like, play, pens, league


In [4]:
# ============================================================
# LSA (SVD)
# ============================================================
lsa = TruncatedSVD(n_components=n_topics, random_state=42)
lsa.fit(X_tfidf)

print("LSA Topics:")
print_topics(lsa, feature_names_tfidf)

LSA Topics:
  Topic 1: like, just, don, people, think, gun, know, space, time, right
  Topic 2: thanks, files, file, image, space, software, 3d, program, mail, advance
  Topic 3: space, nasa, shuttle, launch, moon, orbit, earth, lunar, mars, research
  Topic 4: gun, guns, people, weapons, criminals, crime, law, file, police, control


In [5]:
# ============================================================
# BERTOPIC
# ============================================================
try:
    from bertopic import BERTopic
    from sentence_transformers import SentenceTransformer
    
    # Use a small, fast model
    embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
    
    topic_model = BERTopic(
        embedding_model=embedding_model,
        nr_topics=n_topics,
        min_topic_size=10,
        verbose=False
    )
    
    topics, probs = topic_model.fit_transform(texts)
    
    print("BERTopic Topics:")
    topic_info = topic_model.get_topic_info()
    for _, row in topic_info[topic_info.Topic != -1].head(n_topics).iterrows():
        topic_words = [w for w, _ in topic_model.get_topic(row.Topic)[:8]]
        print(f"  Topic {row.Topic} ({row.Count} docs): {', '.join(topic_words)}")
    
    # Visualize (requires plotly)
    # topic_model.visualize_topics()

except ImportError:
    print("Install: pip install bertopic sentence-transformers")
except Exception as e:
    print(f"BERTopic error: {e}")

Install: pip install bertopic sentence-transformers


In [6]:
# ============================================================
# TOPIC COHERENCE EVALUATION
# ============================================================
try:
    import gensim
    from gensim.models.ldamodel import LdaModel
    from gensim.models.coherencemodel import CoherenceModel
    from gensim import corpora
    import nltk
    from nltk.corpus import stopwords
    nltk.download('stopwords', quiet=True)
    
    stop_words = set(stopwords.words('english'))
    
    # Preprocess
    processed = [
        [w.lower() for w in text.split() if w.lower() not in stop_words and len(w) > 3]
        for text in texts
    ]
    
    dictionary = corpora.Dictionary(processed)
    dictionary.filter_extremes(no_below=2, no_above=0.9)
    corpus_bow = [dictionary.doc2bow(doc) for doc in processed]
    
    # Train LDA with gensim
    lda_gensim = LdaModel(
        corpus=corpus_bow,
        id2word=dictionary,
        num_topics=n_topics,
        random_state=42,
        passes=10
    )
    
    # Coherence
    coherence_model = CoherenceModel(
        model=lda_gensim,
        texts=processed,
        dictionary=dictionary,
        coherence='c_v'
    )
    coherence_score = coherence_model.get_coherence()
    print(f"Topic Coherence (CV): {coherence_score:.4f}")
    
    print("\nGensim LDA Topics:")
    for i, topic in lda_gensim.print_topics(num_words=8):
        print(f"  Topic {i+1}: {topic}")
        
except ImportError:
    print("Install: pip install gensim")

Topic Coherence (CV): 0.3524

Gensim LDA Topics:
  Topic 1: 0.013*"space" + 0.010*"would" + 0.006*"like" + 0.005*"people" + 0.004*"program" + 0.004*"nasa" + 0.004*"shuttle" + 0.003*"telescope"
  Topic 2: 0.009*"would" + 0.005*"like" + 0.005*"points" + 0.005*"team" + 0.004*"pick" + 0.004*"image" + 0.004*"good" + 0.004*"think"
  Topic 3: 0.012*"would" + 0.008*"like" + 0.007*"think" + 0.006*"could" + 0.005*"people" + 0.004*"know" + 0.003*"make" + 0.003*"game"
  Topic 4: 0.006*"first" + 0.006*"space" + 0.005*"probe" + 0.005*"bill" + 0.005*"lunar" + 0.005*"file" + 0.004*"from:" + 0.004*"united"


## Additional Learning Resources

### Papers
- [LDA: Latent Dirichlet Allocation](https://jmlr.org/papers/v3/blei03a.html) Blei, Ng & Jordan, 2003
- [BERTopic: Neural topic modeling with a class-based TF-IDF](https://arxiv.org/abs/2203.05794) Grootendorst, 2022
- [Top2Vec: Distributed Representations of Topics](https://arxiv.org/abs/2008.09470) Angelov, 2020
- [CTM: Contextualized Topic Models](https://arxiv.org/abs/2004.03974) Bianchi et al., 2021

### Libraries & Tools
- [BERTopic Documentation](https://maartengr.github.io/BERTopic/) Comprehensive modern topic modeling
- [gensim LDA](https://radimrehurek.com/gensim/models/ldamodel.html) Fast LDA implementation
- [pyLDAvis](https://github.com/bmabey/pyLDAvis) Interactive LDA visualization
- [Octis](https://github.com/MIND-Lab/OCTIS) Topic model benchmarking

### Tutorials
- [Topic Modeling with Gensim](https://radimrehurek.com/gensim/auto_examples/tutorials/run_lda.html)
- [BERTopic Getting Started](https://maartengr.github.io/BERTopic/getting_started/quickstart/quickstart.html)
- [LDA Visualization with pyLDAvis](https://github.com/bmabey/pyLDAvis)